# Chapter 3 · Vimentin–Microtubule Bonds — Inference on Cytosim Data

Section 3.6 (pp. 51–57), Figures 17–20.

| Figure | Page | Content |
|---|---|---|
| Figure 17 | p. 52 | Raw force curves from the Cytosim simulations |
| Figure 18 | p. 53 | Unbinding statistics |
| Figure 19 | p. 56 | Posterior distributions of the bond parameters |
| Figure 20 | p. 57 | Bond 2 posterior with rejection modelling |

Data: Cytosim simulations by Kristian A. Pajanonot and Komal Bhattacharyya, the authors of these data
(see `data/cytosim/PROVENANCE.md`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import lambert_thesis as lt
from lambert_thesis import cytosim, mcmc, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

The simulations consist of two perpendicular filaments in Cytosim, attached to beads by Hookean springs. The filament
nodes can form a bond with Bell–Evans unbinding; the beads are dragged at velocity $v$, and the force is recorded from
bond formation until the bond breaks. We treat the simulation as a black box, emulating the lab; the ground-truth
bond parameters are known and serve for validation.

Three bond configurations were simulated, each at three polymer stiffnesses $k$ (not part of the unbinding model).
Bonds 1 and 2 were recorded with the censoring protocol (a curve is only kept if its peak force exceeds 2 pN);
bond 3 without it, as a reference.

*Code:* `cytosim.load_csv(path)` returns the force curves (NaN after unbinding), the time axis and the index of the
interval in which each bond broke.

In [ ]:
BONDS = {  # name: (file suffix, ru0, xu, censored)
    "Bond 1": ("1", 0.01, 0.8, True),
    "Bond 2": ("2", 0.15, 0.2, True),
    "Bond 3": ("4_all", 0.15, 0.8, False),
}
STIFFNESSES = [80, 40, 20]
F_THR = 2.0   # pN, censoring threshold

data = {(b, k): cytosim.load_csv(lt.REPO_ROOT / "data" / "cytosim" / f"stiff{k}_{suffix}.csv")
        for b, (suffix, *_) in BONDS.items() for k in STIFFNESSES}
for (b, k), d in data.items():
    d["bond"], d["k"], d["ru0"], d["xu"] = b, k, BONDS[b][1], BONDS[b][2]

def grid_figure(panel, title_fmt, xlabel, ylabel):
    """3x3 grid: rows = bonds, columns = stiffnesses."""
    fig, axes = plt.subplots(3, 3, figsize=(11, 10))
    for r, b in enumerate(BONDS):
        for c, k in enumerate(STIFFNESSES):
            d = data[(b, k)]
            panel(axes[r, c], d)
            axes[r, c].set_title(title_fmt.format(k=k, ru=d["ru0"], xu=d["xu"], thr=F_THR))
        axes[r, 2].annotate(f"{b} ({'censored' if BONDS[b][3] else 'uncensored'})", xy=(1.05, 0.5),
                            xycoords="axes fraction", rotation=90, va="center", fontsize=12,
                            color="red" if BONDS[b][3] else "green")
    for c in range(3):
        axes[0, c].annotate(f"Stiffness {c + 1}", xy=(0.5, 1.2), xycoords="axes fraction", ha="center",
                            fontsize=12, color="tab:orange")
    fig.supxlabel(xlabel); fig.supylabel(ylabel)
    fig.tight_layout()
    return fig, axes

### Figure 17 (p. 52): raw data

In [ ]:
def raw_panel(ax, d):
    for i, f in enumerate(d["F"]):
        ax.plot(d["t"], f + i, color="k")
    for e in d["events"]:
        ax.axvline(d["t"][e], color="tab:blue")

fig, axes = grid_figure(raw_panel, "k={k},ru={ru},xu={xu}, f_thr={thr}", "t [s]", "F [pN]")
axes[0, 0].plot([], [], "k", label="force curve (shifted up by its index)")
axes[0, 0].plot([], [], color="tab:blue", label="unbinding event")
axes[0, 0].legend(loc="upper left", fontsize=7)
savefig("fig17_cytosim_raw_data");

**Figure 17 (p. 52).** Raw data from the Cytosim simulations. Rows: bond configurations (ground truth in the titles);
columns: polymer stiffness. Top and middle rows: censored at 2 pN; bottom row: no censoring. Curves are shifted
upwards to make them discernible (all start at about 0 pN); blue lines mark unbinding events. The force curves show a
softer regime transition than the entropic-spring model and are noisy due to thermal motion. The bond-2 data is cut off
at a maximum simulation time, which piles up unbinding events at the end.

## Unbinding statistics

We compare the histogram of the unbinding times in the data with samples of a model force $\tilde F(t)$ at the ground-truth
parameters. The model force is the mean of all recorded force curves of a data set, continued linearly for as long again.
Besides sampling, the decay of the probability of the bond being intact can be integrated,

$$\frac{dp}{dt}=-p\cdot r_u(\tilde F(t)),\qquad p(t+\Delta t)=p(t)\cdot\exp\left(-\Delta t\,r_{u,0}\exp\left(\frac{x_u\tilde F(t)}{k_BT}\right)\right),$$

normalised using $p(t=0)$; this is more stable than a finite-difference solution.

*Code:* `cytosim.mean_force_curve`, `cytosim.simulate_unbinding_times` (5000 samples), `cytosim.unbinding_time_density`.

In [ ]:
def stats_panel(ax, d):
    model = cytosim.mean_force_curve(d, rel_additional_time=1)
    sim = cytosim.simulate_unbinding_times(model, d["ru0"], d["xu"], 5000, rng)
    ax.hist(d["events_t"], density=True, color="k")
    ax.hist(sim, density=True, color="r", alpha=0.75, bins=25)
    ax.plot(model["t"], cytosim.unbinding_time_density(model, d["ru0"], d["xu"]), "g", lw=2)

fig, axes = grid_figure(stats_panel, "k={k},ru={ru},xu={xu}", "t [s]", "p")
axes[0, 0].hist([], color="k", label="Cytosim data")
axes[0, 0].hist([], color="r", alpha=0.75, label="samples of the model force")
axes[0, 0].plot([], [], "g", lw=2, label="expected statistics (eq. 3.21)")
axes[0, 0].legend(loc="upper left", fontsize=7)
savefig("fig18_unbinding_statistics");

**Figure 18 (p. 53).** Black: unbinding times of the Cytosim data; red: samples of the model force; green: expected
statistics. Bond 1 has a very low base rate, so few unbindings occur at low forces and the censoring hardly matters.
Bond 2 has a high base rate: many unbindings were lost to the censoring protocol, and the data is cut off at a maximum
simulation time. Bond 3 (same physical parameters, no censoring) matches the expected statistics.

## Posterior of the bond parameters

Sampling the posterior confirms the bias caused by the protocol. We sample $p(r_{u,0},x_u\mid\text{data})$ with a flat prior
using a component-wise Metropolis sampler started at the ground truth (800 samples, one stored every 4 iterations,
proposal widths $(4\cdot10^{-3}, 4\cdot10^{-2})$ for bond 1 and ten times larger for bonds 2 and 3). The density is
estimated from the samples (after 100 warm-up samples) with a kernel density estimate; the dashed line encloses 95 %.

*Code:* `cytosim.log_likelihood` (numba), `mcmc.componentwise_metropolis`, `cytosim.posterior_kde`.

In [ ]:
def sample_posterior(d, loglik, n_samples, thin, steps):
    return mcmc.componentwise_metropolis(loglik, [d["ru0"], d["xu"]], n_samples, steps, thin, rng)[0]

def plain_posteriors():
    out = {}
    for (b, k), d in data.items():
        f = 1 if b == "Bond 1" else 10
        s = sample_posterior(d, lambda p, d=d: cytosim.log_likelihood(d, p[0], p[1]), 800, 4, [4e-3 * f, 4e-2 * f])
        out[f"{b}_{k}"] = s
    return out

samples = lt.cached("fig19_cytosim_posterior_samples", plain_posteriors)

def kde_panel(ax, s, d, warm_up=100):
    ru = np.linspace(0, 2 * d["ru0"], 100)
    xu = np.linspace(0, 3 * d["xu"], 100)
    P = cytosim.posterior_kde(s[warm_up:], ru, xu, bandwidth=1.0)
    plotting.grid_posterior(ax, P, ru, xu, xlabel="", ylabel="")
    ax.plot([d["ru0"]], [d["xu"]], "+", color="r", ms=20, mew=3)
    ax.get_legend().remove()

In [ ]:
fig, axes = grid_figure(lambda ax, d: kde_panel(ax, samples[f"{d['bond']}_{d['k']}"], d),
                        "k={k},ru={ru},xu={xu}", "r_u0", "x_u")
axes[0, 0].plot([], [], "r+", ms=10, mew=2, label="ground truth")
axes[0, 0].legend(loc="upper left", fontsize=7)
savefig("fig19_cytosim_posteriors");

**Figure 19 (p. 56).** Posterior distributions of the bond parameters; ground truths in red. Bonds 1 and 2 are subject to
censoring, which biases bond 2 with its high base unbinding rate: the posterior mass shifts to lower base rates, and a
higher sensitivity is needed to explain the (too numerous) unbindings at higher forces. Without censoring, bond 3 recovers
the ground truth.

## Rejection modelling for bond 2

The bias can be removed without losing data by including the threshold in the model (section 3.3.2): each recorded curve
contributes $\mathcal L(T)/(1-P_r)$, where the marginal rejection probability $P_r(r_{u,0},x_u)$ — the probability that the
bond breaks before the force reaches 2 pN — is computed with the mean force curve of the data. Sampler: 3000 samples, one
stored every 100 iterations, proposal widths $(4\cdot10^{-3}, 4\cdot10^{-2})$.

*Code:* `cytosim.log_likelihood_rejection(d, ru0, xu, f_thr)`. Several minutes per stiffness; cached in `results/`.

In [ ]:
def rejection_posteriors():
    out = {}
    for k in STIFFNESSES:
        d = data[("Bond 2", k)]
        out[str(k)] = sample_posterior(d, lambda p, d=d: cytosim.log_likelihood_rejection(d, p[0], p[1], F_THR),
                                       3000, 100, [4e-3, 4e-2])
    return out

samples_rej = lt.cached("fig20_bond2_rejection_samples", rejection_posteriors)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, k in zip(axes, STIFFNESSES):
    d = data[("Bond 2", k)]
    kde_panel(ax, samples_rej[str(k)], d)
    ax.set_title(f"k={k},ru={d['ru0']},xu={d['xu']}")
axes[0].plot([], [], "r+", ms=10, mew=2, label="ground truth")
axes[0].legend(loc="upper left", fontsize=7)
fig.tight_layout()
savefig("fig20_bond2_rejection_modelling");

**Figure 20 (p. 57).** Bond 2 posterior parameters using rejection modelling. The bias is drastically reduced. At the highest
stiffness the ground truth is still not captured, which we attribute to the Cytosim simulation stopping too early (the
cut-off at long times further taints the unbinding statistics).